# Pair-Aware BERTimbau Training

This notebook trains BERTimbau Large with pair-aware supervision on the validated pair-controlled Puntuguese splits. Two training conditions are evaluated: TruePair, which uses each real H/N micro-edited pair, and ShuffledPair, which replaces the true non-pun counterpart with a deterministic different non-pun instance from the same training partition. Both conditions combine instance-level cross-entropy with a logistic pairwise loss. The six split seeds and three model seeds are evaluated under identical hyperparameters, and all models are tested with the same instance-level and true-pair metrics.

In [1]:
from pathlib import Path
import gc
import json
import platform
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers

from IPython.display import display
from sklearn import __version__ as sklearn_version
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging

/home/avelar/pair-aware/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

SPLIT_SEEDS = [13, 21, 40, 42, 73, 101]
MODEL_SEEDS = [40, 123, 456]
PAIRING_STRATEGIES = ["true_pair", "shuffled_pair"]

SHUFFLE_SEED = 2026
PAIR_LOSS_WEIGHT = 1.0

MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
PAIR_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

KEEP_CHECKPOINTS = False
SKIP_COMPLETED_RUNS = True

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_PAIR_COUNTS = {
    "train": 1995,
    "validation": 285,
    "test": 570,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

ID2LABEL = {
    0: "0",
    1: "1",
}

LABEL2ID = {
    "0": 0,
    "1": 1,
}

In [3]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "pair_controlled").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing data/pair_controlled."
    )

In [4]:
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "pair_controlled"
RESULTS_ROOT = PROJECT_ROOT / "results"

for strategy in PAIRING_STRATEGIES:
    (RESULTS_ROOT / strategy).mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:", PROJECT_ROOT)
print("Pair-controlled data:", DATA_DIR)
print("Results root:", RESULTS_ROOT)

Project root: /home/avelar/pair-aware
Pair-controlled data: /home/avelar/pair-aware/data/pair_controlled
Results root: /home/avelar/pair-aware/results


In [5]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn_version)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)
print("Split seeds:", SPLIT_SEEDS)
print("Model seeds:", MODEL_SEEDS)
print("Pairing strategies:", PAIRING_STRATEGIES)
print("Pair loss weight:", PAIR_LOSS_WEIGHT)
print("Pair batch size:", PAIR_BATCH_SIZE)
print("Effective texts per training batch:", PAIR_BATCH_SIZE * 2)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("cuDNN:", torch.backends.cudnn.version())

Python: 3.12.14
Platform: Linux-7.0.0-28-generic-x86_64-with-glibc2.43
PyTorch: 2.13.0+cu130
Transformers: 5.15.0
NumPy: 2.5.3
Pandas: 2.3.2
scikit-learn: 1.9.0
CUDA available: True
CUDA: 13.0
Split seeds: [13, 21, 40, 42, 73, 101]
Model seeds: [40, 123, 456]
Pairing strategies: ['true_pair', 'shuffled_pair']
Pair loss weight: 1.0
Pair batch size: 4
Effective texts per training batch: 8
GPU: NVIDIA GeForce RTX 3060
cuDNN: 92000


In [6]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

if DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA is required for the planned BERTimbau Large experiments."
    )

test_tensor = torch.randn(
    64,
    64,
    device=DEVICE,
)

test_result = test_tensor @ test_tensor

print("Device:", DEVICE)
print("CUDA computation:", test_result.device)

Device: cuda
CUDA computation: cuda:0


In [7]:
def set_model_seed(model_seed):
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(model_seed)
        torch.cuda.manual_seed_all(model_seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [8]:
def read_jsonl(file_path):
    rows = []

    with Path(file_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return pd.DataFrame(rows)

In [9]:
def parse_pair_id(example_id):
    parts = str(example_id).rsplit(".", 1)

    if len(parts) != 2:
        raise ValueError(
            f"Invalid example ID: {example_id}"
        )

    pair_id, suffix = parts

    if not pair_id or suffix not in {"H", "N"}:
        raise ValueError(
            f"Invalid example ID: {example_id}"
        )

    return pair_id, suffix

In [10]:
def load_split_directory(split_dir):
    split_dir = Path(split_dir)

    split_data = {}

    for split_name in ["train", "validation", "test"]:
        file_path = split_dir / f"{split_name}.jsonl"

        if not file_path.is_file():
            raise FileNotFoundError(
                f"Missing split file: {file_path}"
            )

        split_data[split_name] = read_jsonl(
            file_path
        )

    return split_data

In [11]:
def validate_input_splits(split_data, run_name):
    pair_sets = {}
    all_ids = []

    for split_name in ["train", "validation", "test"]:
        split_df = split_data[split_name].copy()

        required_columns = {
            "id",
            "text",
            "label",
        }

        missing_columns = required_columns - set(
            split_df.columns
        )

        if missing_columns:
            raise ValueError(
                f"{run_name}/{split_name}: missing columns "
                f"{sorted(missing_columns)}."
            )

        if len(split_df) != EXPECTED_SPLIT_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{EXPECTED_SPLIT_COUNTS[split_name]} examples, "
                f"found {len(split_df)}."
            )

        split_df["label"] = split_df["label"].astype(int)

        observed_classes = (
            split_df["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed_classes != EXPECTED_CLASS_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected class distribution "
                f"{EXPECTED_CLASS_COUNTS[split_name]}, "
                f"found {observed_classes}."
            )

        if split_df["id"].duplicated().any():
            raise ValueError(
                f"{run_name}/{split_name}: duplicated IDs."
            )

        pair_members = {}

        for example_id in split_df["id"].astype(str):
            pair_id, suffix = parse_pair_id(example_id)
            pair_members.setdefault(pair_id, []).append(suffix)

        invalid_pairs = {
            pair_id: suffixes
            for pair_id, suffixes in pair_members.items()
            if sorted(suffixes) != ["H", "N"]
        }

        if invalid_pairs:
            raise ValueError(
                f"{run_name}/{split_name}: invalid H/N pairs found."
            )

        if len(pair_members) != EXPECTED_PAIR_COUNTS[split_name]:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{EXPECTED_PAIR_COUNTS[split_name]} pairs, "
                f"found {len(pair_members)}."
            )

        pair_sets[split_name] = set(
            pair_members
        )

        all_ids.extend(
            split_df["id"]
            .astype(str)
            .tolist()
        )

    if len(all_ids) != 5700:
        raise ValueError(
            f"{run_name}: expected 5700 total IDs."
        )

    if len(set(all_ids)) != 5700:
        raise ValueError(
            f"{run_name}: IDs overlap across splits."
        )

    crossing_pairs = (
        pair_sets["train"].intersection(pair_sets["validation"])
        | pair_sets["train"].intersection(pair_sets["test"])
        | pair_sets["validation"].intersection(pair_sets["test"])
    )

    if crossing_pairs:
        raise ValueError(
            f"{run_name}: pair IDs overlap across splits."
        )

In [12]:
def build_true_pairs(dataframe):
    working = dataframe.copy()

    parsed = working["id"].astype(str).map(
        parse_pair_id
    )

    working["pair_id"] = parsed.map(
        lambda value: value[0]
    )

    working["suffix"] = parsed.map(
        lambda value: value[1]
    )

    rows = []

    for pair_id, pair_df in working.groupby(
        "pair_id",
        sort=True,
    ):
        if set(pair_df["suffix"]) != {"H", "N"}:
            raise ValueError(
                f"Pair {pair_id} does not contain one H and one N instance."
            )

        pun_row = pair_df.loc[
            pair_df["suffix"] == "H"
        ].iloc[0]

        non_pun_row = pair_df.loc[
            pair_df["suffix"] == "N"
        ].iloc[0]

        rows.append(
            {
                "pair_id": pair_id,
                "pun_id": str(pun_row["id"]),
                "pun_text": str(pun_row["text"]),
                "non_pun_id": str(non_pun_row["id"]),
                "non_pun_text": str(non_pun_row["text"]),
            }
        )

    return pd.DataFrame(rows)

In [13]:
def sattolo_permutation(length, seed):
    if length < 2:
        raise ValueError(
            "A derangement requires at least two elements."
        )

    rng = random.Random(seed)
    permutation = list(range(length))

    for index in range(length - 1, 0, -1):
        selected = rng.randrange(index)
        permutation[index], permutation[selected] = (
            permutation[selected],
            permutation[index],
        )

    if any(
        index == value
        for index, value in enumerate(permutation)
    ):
        raise RuntimeError(
            "The generated permutation contains fixed points."
        )

    return permutation

In [14]:
def build_shuffled_pairs(true_pairs, split_seed):
    shuffled = true_pairs.copy().reset_index(
        drop=True
    )

    permutation = sattolo_permutation(
        length=len(shuffled),
        seed=SHUFFLE_SEED + split_seed,
    )

    non_pun_ids = shuffled["non_pun_id"].tolist()
    non_pun_texts = shuffled["non_pun_text"].tolist()
    original_pair_ids = shuffled["pair_id"].tolist()

    shuffled["non_pun_id"] = [
        non_pun_ids[index]
        for index in permutation
    ]

    shuffled["non_pun_text"] = [
        non_pun_texts[index]
        for index in permutation
    ]

    shuffled["non_pun_source_pair_id"] = [
        original_pair_ids[index]
        for index in permutation
    ]

    if (
        shuffled["pair_id"]
        == shuffled["non_pun_source_pair_id"]
    ).any():
        raise RuntimeError(
            "ShuffledPair contains a true counterpart."
        )

    if len(
        set(shuffled["non_pun_id"])
    ) != len(shuffled):
        raise RuntimeError(
            "ShuffledPair does not use each non-pun instance exactly once."
        )

    return shuffled

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer loaded:",
    MODEL_NAME,
)

Tokenizer loaded: neuralmind/bert-large-portuguese-cased


In [16]:
class PairDataset(Dataset):
    def __init__(
        self,
        pairs,
        tokenizer,
        max_length,
    ):
        self.pun_texts = (
            pairs["pun_text"]
            .astype(str)
            .tolist()
        )

        self.non_pun_texts = (
            pairs["non_pun_text"]
            .astype(str)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(
            self.pun_texts
        )

    def encode(self, text):
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

    def __getitem__(self, index):
        pun_encoding = self.encode(
            self.pun_texts[index]
        )

        non_pun_encoding = self.encode(
            self.non_pun_texts[index]
        )

        item = {}

        for key, value in pun_encoding.items():
            item[f"pun_{key}"] = value

        for key, value in non_pun_encoding.items():
            item[f"non_pun_{key}"] = value

        return item

In [17]:
class InstanceDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
    ):
        self.texts = (
            dataframe["text"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(
            self.texts
        )

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long,
        )

        return item

In [18]:
def create_pair_dataloader(
    pairs,
    model_seed,
):
    dataset = PairDataset(
        pairs=pairs,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    generator = torch.Generator()
    generator.manual_seed(
        model_seed
    )

    return DataLoader(
        dataset,
        batch_size=PAIR_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
    )

In [19]:
def create_instance_dataloader(
    dataframe,
):
    dataset = InstanceDataset(
        dataframe=dataframe,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

In [20]:
def move_model_inputs_to_device(
    batch,
    prefix,
):
    inputs = {}

    for key, value in batch.items():
        if key.startswith(prefix):
            clean_key = key[len(prefix):]
            inputs[clean_key] = value.to(
                DEVICE,
                non_blocking=True,
            )

    return inputs

In [21]:
def load_fresh_model(model_seed):
    set_model_seed(
        model_seed
    )

    previous_verbosity = (
        hf_logging.get_verbosity()
    )

    hf_logging.set_verbosity_error()

    try:
        model = (
            AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=2,
                id2label=ID2LABEL,
                label2id=LABEL2ID,
            )
        )
    finally:
        hf_logging.set_verbosity(
            previous_verbosity
        )

    return model.to(
        DEVICE
    )

In [22]:
def compute_pair_aware_loss(
    model,
    batch,
):
    pun_inputs = move_model_inputs_to_device(
        batch,
        "pun_",
    )

    non_pun_inputs = move_model_inputs_to_device(
        batch,
        "non_pun_",
    )

    combined_inputs = {}

    for key in pun_inputs:
        combined_inputs[key] = torch.cat(
            [
                pun_inputs[key],
                non_pun_inputs[key],
            ],
            dim=0,
        )

    outputs = model(
        **combined_inputs
    )

    logits = outputs.logits

    batch_size = pun_inputs["input_ids"].size(
        0
    )

    pun_logits = logits[
        :batch_size
    ]

    non_pun_logits = logits[
        batch_size:
    ]

    labels = torch.cat(
        [
            torch.ones(
                batch_size,
                dtype=torch.long,
                device=DEVICE,
            ),
            torch.zeros(
                batch_size,
                dtype=torch.long,
                device=DEVICE,
            ),
        ],
        dim=0,
    )

    classification_loss = F.cross_entropy(
        logits,
        labels,
    )

    pun_scores = (
        pun_logits[:, 1]
        - pun_logits[:, 0]
    )

    non_pun_scores = (
        non_pun_logits[:, 1]
        - non_pun_logits[:, 0]
    )

    pair_loss = F.softplus(
        -(
            pun_scores
            - non_pun_scores
        )
    ).mean()

    total_loss = (
        classification_loss
        + PAIR_LOSS_WEIGHT
        * pair_loss
    )

    return (
        total_loss,
        classification_loss,
        pair_loss,
    )

In [23]:
def predict_instances(
    model,
    dataloader,
):
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch.pop(
                "labels"
            )

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True,
                )
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(
                    **inputs
                )

            all_logits.append(
                outputs.logits.detach().cpu()
            )

            all_labels.append(
                labels.cpu()
            )

    logits = torch.cat(
        all_logits,
        dim=0,
    ).numpy()

    labels = torch.cat(
        all_labels,
        dim=0,
    ).numpy()

    return logits, labels

In [24]:
def compute_instance_metrics(
    labels,
    logits,
):
    predictions = np.argmax(
        logits,
        axis=1,
    )

    return {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "precision_macro": float(
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
    }

In [25]:
def train_pair_aware_model(
    model,
    train_loader,
    validation_loader,
    checkpoint_path,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = (
        len(train_loader)
        * NUM_EPOCHS
    )

    warmup_steps = int(
        total_steps
        * WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    best_validation_f1 = -np.inf
    best_epoch = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(
        1,
        NUM_EPOCHS + 1,
    ):
        model.train()

        total_loss_sum = 0.0
        classification_loss_sum = 0.0
        pair_loss_sum = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch}/{NUM_EPOCHS}",
            leave=False,
        )

        for batch in progress:
            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                (
                    total_loss,
                    classification_loss,
                    pair_loss,
                ) = compute_pair_aware_loss(
                    model,
                    batch,
                )

            scaler.scale(
                total_loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            scaler.step(
                optimizer
            )

            scaler.update()
            scheduler.step()

            total_loss_sum += float(
                total_loss.detach().cpu()
            )

            classification_loss_sum += float(
                classification_loss.detach().cpu()
            )

            pair_loss_sum += float(
                pair_loss.detach().cpu()
            )

            progress.set_postfix(
                loss=f"{total_loss_sum / max(1, progress.n + 1):.4f}"
            )

        validation_logits, validation_labels = predict_instances(
            model,
            validation_loader,
        )

        validation_metrics = compute_instance_metrics(
            validation_labels,
            validation_logits,
        )

        epoch_result = {
            "epoch": epoch,
            "train_total_loss": total_loss_sum / len(train_loader),
            "train_classification_loss": classification_loss_sum / len(train_loader),
            "train_pair_loss": pair_loss_sum / len(train_loader),
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_f1_macro": validation_metrics["f1_macro"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }

        history.append(
            epoch_result
        )

        print(
            f"Epoch {epoch}: "
            f"train_loss={epoch_result['train_total_loss']:.6f} "
            f"validation_f1_macro={epoch_result['validation_f1_macro']:.6f}"
        )

        if (
            validation_metrics["f1_macro"]
            > best_validation_f1
        ):
            best_validation_f1 = (
                validation_metrics["f1_macro"]
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            break

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=DEVICE,
        )
    )

    return {
        "history": history,
        "best_validation_f1_macro": float(
            best_validation_f1
        ),
        "best_epoch": int(
            best_epoch
        ),
        "total_training_steps": int(
            total_steps
        ),
        "warmup_steps": int(
            warmup_steps
        ),
    }

In [26]:
def softmax_numpy(logits):
    shifted = logits - np.max(
        logits,
        axis=1,
        keepdims=True,
    )

    exponentials = np.exp(
        shifted
    )

    return exponentials / np.sum(
        exponentials,
        axis=1,
        keepdims=True,
    )

In [27]:
def evaluate_instance_predictions(
    labels,
    logits,
):
    predictions = np.argmax(
        logits,
        axis=1,
    )

    report_dict = classification_report(
        labels,
        predictions,
        labels=[0, 1],
        target_names=["non_pun", "pun"],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "precision_non_pun": float(
            report_dict["non_pun"]["precision"]
        ),
        "recall_non_pun": float(
            report_dict["non_pun"]["recall"]
        ),
        "f1_non_pun": float(
            report_dict["non_pun"]["f1-score"]
        ),
        "precision_pun": float(
            report_dict["pun"]["precision"]
        ),
        "recall_pun": float(
            report_dict["pun"]["recall"]
        ),
        "f1_pun": float(
            report_dict["pun"]["f1-score"]
        ),
        "precision_macro": float(
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_weighted": float(
            precision_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "recall_weighted": float(
            recall_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }

    return (
        metrics,
        report_dict,
        cm,
        predictions,
    )

In [28]:
def build_instance_predictions(
    test_df,
    logits,
):
    probabilities = softmax_numpy(
        logits
    )

    predictions = np.argmax(
        logits,
        axis=1,
    )

    rows = []

    for index, row in test_df.reset_index(
        drop=True
    ).iterrows():
        pair_id, suffix = parse_pair_id(
            row["id"]
        )

        rows.append(
            {
                "id": str(row["id"]),
                "pair_id": pair_id,
                "suffix": suffix,
                "true_label": int(row["label"]),
                "predicted_label": int(
                    predictions[index]
                ),
                "logit_non_pun": float(
                    logits[index, 0]
                ),
                "logit_pun": float(
                    logits[index, 1]
                ),
                "pun_score": float(
                    logits[index, 1]
                    - logits[index, 0]
                ),
                "probability_non_pun": float(
                    probabilities[index, 0]
                ),
                "probability_pun": float(
                    probabilities[index, 1]
                ),
            }
        )

    return pd.DataFrame(
        rows
    )

In [29]:
def evaluate_true_pairs(
    instance_predictions,
):
    pair_rows = []

    grouped = instance_predictions.groupby(
        "pair_id",
        sort=True,
    )

    for pair_id, pair_df in grouped:
        if len(pair_df) != 2:
            raise ValueError(
                f"Pair {pair_id} does not contain exactly two instances."
            )

        if set(pair_df["suffix"]) != {"H", "N"}:
            raise ValueError(
                f"Pair {pair_id} does not contain one H and one N instance."
            )

        pun_row = pair_df.loc[
            pair_df["suffix"] == "H"
        ].iloc[0]

        non_pun_row = pair_df.loc[
            pair_df["suffix"] == "N"
        ].iloc[0]

        margin = float(
            pun_row["pun_score"]
            - non_pun_row["pun_score"]
        )

        ranking_correct = bool(
            margin > 0
        )

        ranking_tie = bool(
            margin == 0
        )

        exact_match = bool(
            pun_row["predicted_label"] == 1
            and non_pun_row["predicted_label"] == 0
        )

        pair_rows.append(
            {
                "pair_id": pair_id,
                "pun_id": pun_row["id"],
                "non_pun_id": non_pun_row["id"],
                "pun_predicted_label": int(
                    pun_row["predicted_label"]
                ),
                "non_pun_predicted_label": int(
                    non_pun_row["predicted_label"]
                ),
                "pun_score": float(
                    pun_row["pun_score"]
                ),
                "non_pun_score": float(
                    non_pun_row["pun_score"]
                ),
                "pun_probability": float(
                    pun_row["probability_pun"]
                ),
                "non_pun_probability": float(
                    non_pun_row["probability_pun"]
                ),
                "pair_margin": margin,
                "ranking_correct": ranking_correct,
                "ranking_tie": ranking_tie,
                "exact_match": exact_match,
            }
        )

    pair_predictions = pd.DataFrame(
        pair_rows
    )

    metrics = {
        "pair_count": int(
            len(pair_predictions)
        ),
        "pair_accuracy": float(
            pair_predictions["ranking_correct"].mean()
        ),
        "pair_exact_match": float(
            pair_predictions["exact_match"].mean()
        ),
        "pair_ties": int(
            pair_predictions["ranking_tie"].sum()
        ),
        "mean_pair_margin": float(
            pair_predictions["pair_margin"].mean()
        ),
        "median_pair_margin": float(
            pair_predictions["pair_margin"].median()
        ),
        "std_pair_margin": float(
            pair_predictions["pair_margin"].std(ddof=1)
        ),
        "min_pair_margin": float(
            pair_predictions["pair_margin"].min()
        ),
        "max_pair_margin": float(
            pair_predictions["pair_margin"].max()
        ),
    }

    return (
        metrics,
        pair_predictions,
    )

In [30]:
def save_run_outputs(
    output_dir,
    metrics,
    report_dict,
    cm,
    metadata,
    history,
    instance_predictions,
    pair_predictions,
):
    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        cm,
        index=[
            "true_non_pun",
            "true_pun",
        ],
        columns=[
            "pred_non_pun",
            "pred_pun",
        ],
    ).to_csv(
        output_dir / "confusion_matrix.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        report_dict
    ).T.to_csv(
        output_dir / "classification_report.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        history
    ).to_csv(
        output_dir / "training_history.csv",
        index=False,
        encoding="utf-8",
    )

    instance_predictions.to_csv(
        output_dir / "instance_predictions.csv",
        index=False,
        encoding="utf-8",
    )

    pair_predictions.to_csv(
        output_dir / "pair_predictions.csv",
        index=False,
        encoding="utf-8",
    )

    with (
        output_dir / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    with (
        output_dir / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

In [31]:
def load_existing_run(
    output_dir,
):
    output_dir = Path(
        output_dir
    )

    metrics_path = (
        output_dir
        / "metrics.json"
    )

    metadata_path = (
        output_dir
        / "metadata.json"
    )

    if not (
        metrics_path.is_file()
        and metadata_path.is_file()
    ):
        return None

    with metrics_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(
            file
        )

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(
            file
        )

    return {
        "method": metadata["method"],
        "split_seed": metadata["split_seed"],
        "model_seed": metadata["model_seed"],
        **metrics,
    }

In [32]:
def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [33]:
def run_pair_aware(
    pairing_strategy,
    split_seed,
    model_seed,
):
    if pairing_strategy not in PAIRING_STRATEGIES:
        raise ValueError(
            f"Unknown pairing strategy: {pairing_strategy}"
        )

    split_dir = (
        DATA_DIR
        / f"seed_{split_seed}"
    )

    output_dir = (
        RESULTS_ROOT
        / pairing_strategy
        / f"split_{split_seed}"
        / f"model_seed_{model_seed}"
    )

    if SKIP_COMPLETED_RUNS:
        existing_result = load_existing_run(
            output_dir
        )

        if existing_result is not None:
            print(
                "Skipping completed run:",
                pairing_strategy,
                f"split={split_seed}",
                f"model_seed={model_seed}",
            )

            return existing_result

    start_time = time.time()

    set_model_seed(
        model_seed
    )

    split_data = load_split_directory(
        split_dir
    )

    validate_input_splits(
        split_data,
        f"{pairing_strategy}/split_{split_seed}",
    )

    train_df = split_data["train"]
    validation_df = split_data["validation"]
    test_df = split_data["test"]

    true_train_pairs = build_true_pairs(
        train_df
    )

    if pairing_strategy == "true_pair":
        training_pairs = true_train_pairs
    else:
        training_pairs = build_shuffled_pairs(
            true_pairs=true_train_pairs,
            split_seed=split_seed,
        )

    train_loader = create_pair_dataloader(
        pairs=training_pairs,
        model_seed=model_seed,
    )

    validation_loader = create_instance_dataloader(
        validation_df
    )

    test_loader = create_instance_dataloader(
        test_df
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_path = (
        output_dir
        / "best_model.pt"
    )

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    model = load_fresh_model(
        model_seed
    )

    training_result = train_pair_aware_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        checkpoint_path=checkpoint_path,
    )

    test_logits, test_labels = predict_instances(
        model,
        test_loader,
    )

    (
        instance_metrics,
        report_dict,
        cm,
        predictions,
    ) = evaluate_instance_predictions(
        labels=test_labels,
        logits=test_logits,
    )

    instance_predictions = build_instance_predictions(
        test_df=test_df,
        logits=test_logits,
    )

    (
        pair_metrics,
        pair_predictions,
    ) = evaluate_true_pairs(
        instance_predictions
    )

    metrics = {
        **instance_metrics,
        **pair_metrics,
        "best_validation_f1_macro": training_result[
            "best_validation_f1_macro"
        ],
        "best_epoch": training_result[
            "best_epoch"
        ],
        "total_runtime": float(
            time.time() - start_time
        ),
    }

    metadata = {
        "method": pairing_strategy,
        "model": MODEL_NAME,
        "split_seed": split_seed,
        "model_seed": model_seed,
        "shuffle_seed": (
            SHUFFLE_SEED
            if pairing_strategy == "shuffled_pair"
            else None
        ),
        "effective_shuffle_seed": (
            SHUFFLE_SEED + split_seed
            if pairing_strategy == "shuffled_pair"
            else None
        ),
        "pair_loss_weight": PAIR_LOSS_WEIGHT,
        "pair_loss": "softplus(-(score_H-score_N))",
        "pun_score": "logit_pun-logit_non_pun",
        "classification_loss": "cross_entropy",
        "train_examples": int(
            len(train_df)
        ),
        "train_pairs": int(
            len(training_pairs)
        ),
        "validation_examples": int(
            len(validation_df)
        ),
        "test_examples": int(
            len(test_df)
        ),
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "pair_batch_size": PAIR_BATCH_SIZE,
        "effective_texts_per_training_batch": PAIR_BATCH_SIZE * 2,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": training_result[
            "warmup_steps"
        ],
        "max_grad_norm": MAX_GRAD_NORM,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "metric_for_best_model": "validation_f1_macro",
        "best_validation_f1_macro": training_result[
            "best_validation_f1_macro"
        ],
        "best_epoch": training_result[
            "best_epoch"
        ],
        "fp16": USE_AMP,
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn_version,
        "cuda_available": bool(
            torch.cuda.is_available()
        ),
        "cuda_version": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(
            0
        ),
        "cudnn": torch.backends.cudnn.version(),
        "checkpoints_retained": KEEP_CHECKPOINTS,
    }

    save_run_outputs(
        output_dir=output_dir,
        metrics=metrics,
        report_dict=report_dict,
        cm=cm,
        metadata=metadata,
        history=training_result["history"],
        instance_predictions=instance_predictions,
        pair_predictions=pair_predictions,
    )

    print("=" * 80)
    print("Method:", pairing_strategy)
    print("Split seed:", split_seed)
    print("Model seed:", model_seed)

    print(
        classification_report(
            test_labels,
            predictions,
            labels=[0, 1],
            target_names=["non_pun", "pun"],
            digits=4,
            zero_division=0,
        )
    )

    print(
        "Accuracy:",
        f"{metrics['accuracy']:.6f}",
    )

    print(
        "Macro-F1:",
        f"{metrics['f1_macro']:.6f}",
    )

    print(
        "Pair Accuracy:",
        f"{metrics['pair_accuracy']:.6f}",
    )

    print(
        "Pair Exact Match:",
        f"{metrics['pair_exact_match']:.6f}",
    )

    print(
        "Mean Pair Margin:",
        f"{metrics['mean_pair_margin']:.6f}",
    )

    result = {
        "method": pairing_strategy,
        "split_seed": split_seed,
        "model_seed": model_seed,
        **metrics,
    }

    if not KEEP_CHECKPOINTS:
        checkpoint_path.unlink(
            missing_ok=True
        )

    del predictions
    del test_logits
    del test_labels
    del train_loader
    del validation_loader
    del test_loader
    del model

    clear_memory()

    return result

In [ ]:
all_results = []

for pairing_strategy in PAIRING_STRATEGIES:
    for model_seed in MODEL_SEEDS:
        for split_seed in SPLIT_SEEDS:
            result = run_pair_aware(
                pairing_strategy=pairing_strategy,
                split_seed=split_seed,
                model_seed=model_seed,
            )

            all_results.append(
                result
            )

all_results_df = pd.DataFrame(
    all_results
).sort_values(
    [
        "method",
        "split_seed",
        "model_seed",
    ]
).reset_index(
    drop=True
)

display(
    all_results_df
)

for pairing_strategy in PAIRING_STRATEGIES:
    strategy_results = all_results_df.loc[
        all_results_df["method"]
        == pairing_strategy
    ]

    strategy_results.to_csv(
        RESULTS_ROOT
        / pairing_strategy
        / "runs.csv",
        index=False,
        encoding="utf-8",
    )

Epoch 1/6:   0%|          | 0/499 [00:00<?, ?it/s]/tmp/ipykernel_202178/692529735.py:92: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1: train_loss=1.115983 validation_f1_macro=0.761802


Epoch 2: train_loss=0.747728 validation_f1_macro=0.807018


Epoch 3: train_loss=0.521741 validation_f1_macro=0.753030


Epoch 4: train_loss=0.361584 validation_f1_macro=0.780902
Method: true_pair
Split seed: 13
Model seed: 40
              precision    recall  f1-score   support

     non_pun     0.7738    0.7561    0.7649       570
         pun     0.7616    0.7789    0.7702       570

    accuracy                         0.7675      1140
   macro avg     0.7677    0.7675    0.7675      1140
weighted avg     0.7677    0.7675    0.7675      1140

Accuracy: 0.767544
Macro-F1: 0.767514
Pair Accuracy: 0.845614
Pair Exact Match: 0.566667
Mean Pair Margin: 3.380870


...

Epoch 3: train_loss=0.486571 validation_f1_macro=0.742468


Epoch 4: train_loss=0.379610 validation_f1_macro=0.764422


Epoch 5: train_loss=0.274089 validation_f1_macro=0.749572


Epoch 6: train_loss=0.184956 validation_f1_macro=0.765569
Method: true_pair
Split seed: 73
Model seed: 40
              precision    recall  f1-score   support

     non_pun     0.7183    0.8456    0.7768       570
         pun     0.8124    0.6684    0.7334       570

    accuracy                         0.7570      1140
   macro avg     0.7653    0.7570    0.7551      1140
weighted avg     0.7653    0.7570    0.7551      1140

Accuracy: 0.757018
Macro-F1: 0.755095
Pair Accuracy: 0.847368
Pair Exact Match: 0.550877
Mean Pair Margin: 8.329042


Epoch 1/6:   0%|          | 0/499 [00:00<?, ?it/s]/tmp/ipykernel_202178/692529735.py:92: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "pair_accuracy",
    "pair_exact_match",
    "mean_pair_margin",
    "median_pair_margin",
    "std_pair_margin",
    "tp",
    "tn",
    "fp",
    "fn",
]

In [ ]:
def build_strategy_summaries(
    results_df,
    strategy,
):
    strategy_df = results_df.loc[
        results_df["method"]
        == strategy
    ].copy()

    split_rows = []

    for split_seed in SPLIT_SEEDS:
        split_df = strategy_df.loc[
            strategy_df["split_seed"]
            == split_seed
        ]

        row = {
            "method": strategy,
            "split_seed": split_seed,
            "model_seed_runs": len(
                split_df
            ),
        }

        for metric in SUMMARY_METRICS:
            row[
                f"{metric}_mean"
            ] = float(
                split_df[metric].mean()
            )

            row[
                f"{metric}_std"
            ] = float(
                split_df[metric].std(ddof=1)
            )

        split_rows.append(
            row
        )

    split_summary = pd.DataFrame(
        split_rows
    )

    model_seed_rows = []

    for model_seed in MODEL_SEEDS:
        model_seed_df = strategy_df.loc[
            strategy_df["model_seed"]
            == model_seed
        ]

        row = {
            "method": strategy,
            "model_seed": model_seed,
            "split_runs": len(
                model_seed_df
            ),
        }

        for metric in SUMMARY_METRICS:
            row[
                f"{metric}_mean"
            ] = float(
                model_seed_df[metric].mean()
            )

            row[
                f"{metric}_std"
            ] = float(
                model_seed_df[metric].std(ddof=1)
            )

        model_seed_rows.append(
            row
        )

    model_seed_summary = pd.DataFrame(
        model_seed_rows
    )

    overall = {
        "method": strategy,
        "split_count": len(
            SPLIT_SEEDS
        ),
        "model_seed_count": len(
            MODEL_SEEDS
        ),
        "total_runs": len(
            strategy_df
        ),
    }

    for metric in SUMMARY_METRICS:
        split_metric = split_summary[
            f"{metric}_mean"
        ]

        overall[
            f"{metric}_mean"
        ] = float(
            split_metric.mean()
        )

        overall[
            f"{metric}_std"
        ] = float(
            split_metric.std(ddof=1)
        )

    overall_summary = pd.DataFrame(
        [overall]
    )

    return (
        split_summary,
        model_seed_summary,
        overall_summary,
    )

In [ ]:
overall_summaries = []

for pairing_strategy in PAIRING_STRATEGIES:
    (
        split_summary,
        model_seed_summary,
        overall_summary,
    ) = build_strategy_summaries(
        results_df=all_results_df,
        strategy=pairing_strategy,
    )

    strategy_dir = (
        RESULTS_ROOT
        / pairing_strategy
    )

    split_summary.to_csv(
        strategy_dir
        / "summary_by_split.csv",
        index=False,
        encoding="utf-8",
    )

    model_seed_summary.to_csv(
        strategy_dir
        / "summary_by_model_seed.csv",
        index=False,
        encoding="utf-8",
    )

    overall_summary.to_csv(
        strategy_dir
        / "summary_overall.csv",
        index=False,
        encoding="utf-8",
    )

    overall_summaries.append(
        overall_summary
    )

    print(
        pairing_strategy
    )

    display(
        overall_summary
    )

pair_aware_summary_df = pd.concat(
    overall_summaries,
    ignore_index=True,
)

display(
    pair_aware_summary_df
)

pair_aware_summary_df.to_csv(
    RESULTS_ROOT
    / "pair_aware_summary.csv",
    index=False,
    encoding="utf-8",
)